In [ ]:
import pandas as pd  
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('data/transaction_dataset.csv')
# df.drop(columns=["isFlaggedFraud"], inplace=True)
df.head(5)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [ ]:
def audit_dataset(df):
    """Print the main PaySim dataset audit results."""

    print("\n" + "=" * 60)
    print("PAYSIM DATASET AUDIT")
    print("=" * 60)

    # Dataset shape
    print("\n1. DATASET SHAPE")
    print(f"Rows:    {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")

    # Column data types
    print("\n2. COLUMN DATA TYPES")
    print(df.dtypes.to_string())

    # Missing values
    missing_values = df.isna().sum()
    missing_percentages = (missing_values / len(df)) * 100

    missing_report = pd.DataFrame(
        {
            "missing_count": missing_values,
            "missing_percentage": missing_percentages,
        }
    )

    print("\n3. MISSING VALUES")
    print(missing_report.to_string())

    # Duplicate rows
    duplicate_count = int(df.duplicated().sum())
    duplicate_percentage = (duplicate_count / len(df)) * 100

    print("\n4. DUPLICATE ROWS")
    print(f"Duplicate rows:       {duplicate_count:,}")
    print(f"Duplicate percentage: {duplicate_percentage:.6f}%")

    # Fraud distribution
    fraud_counts = df["isFraud"].value_counts().sort_index()
    fraud_percentages = (
        df["isFraud"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
    )

    fraud_report = pd.DataFrame(
        {
            "transaction_count": fraud_counts,
            "percentage": fraud_percentages,
        }
    )

    fraud_report.index = fraud_report.index.map(
        {
            0: "Non-fraud",
            1: "Fraud",
        }
    )

    print("\n5. FRAUD DISTRIBUTION")
    print(fraud_report.to_string())

    fraud_count = int(df["isFraud"].sum())
    fraud_percentage = (fraud_count / len(df)) * 100

    print(f"\nTotal fraudulent transactions: {fraud_count:,}")
    print(f"Fraud percentage:              {fraud_percentage:.6f}%")

    # Transaction-type distribution
    type_counts = df["type"].value_counts()
    type_percentages = df["type"].value_counts(normalize=True).mul(100)

    transaction_type_report = pd.DataFrame(
        {
            "transaction_count": type_counts,
            "percentage": type_percentages,
        }
    )

    print("\n6. TRANSACTION-TYPE DISTRIBUTION")
    print(transaction_type_report.to_string())

    print("\n" + "=" * 60)
    print("AUDIT COMPLETED")
    print("=" * 60)


In [ ]:
audit_dataset(df)


PAYSIM DATASET AUDIT

1. DATASET SHAPE
Rows:    6,362,620
Columns: 11

2. COLUMN DATA TYPES
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64

3. MISSING VALUES
                missing_count  missing_percentage
step                        0                 0.0
type                        0                 0.0
amount                      0                 0.0
nameOrig                    0                 0.0
oldbalanceOrg               0                 0.0
newbalanceOrig              0                 0.0
nameDest                    0                 0.0
oldbalanceDest              0                 0.0
newbalanceDest              0                 0.0
isFraud                     0                 0.0
isFlaggedFraud              0                 0.0

4

In [ ]:
def analyse_fraud(df):
    print("\n" + "=" * 70)
    print("PAYSIM FRAUD ANALYSIS")
    print("=" * 70)

    # Fraud by transaction type
    fraud_by_type = pd.crosstab(
        df["type"],
        df["isFraud"],
        margins=True,
    )

    fraud_by_type = fraud_by_type.rename(
        columns={
            0: "non_fraud",
            1: "fraud",
        }
    )

    print("\n1. FRAUD COUNTS BY TRANSACTION TYPE")
    print(fraud_by_type)

    # Fraud percentage within each transaction type
    type_summary = (
        df.groupby("type", observed=True)["isFraud"]
        .agg(
            total_transactions="count",
            fraud_transactions="sum",
        )
        .sort_values("fraud_transactions", ascending=False)
    )

    type_summary["fraud_percentage"] = (
        type_summary["fraud_transactions"]
        / type_summary["total_transactions"]
        * 100
    )

    print("\n2. FRAUD RATE BY TRANSACTION TYPE")
    print(type_summary)

    # Existing fraud flag comparison
    flag_comparison = pd.crosstab(
        df["isFraud"],
        df["isFlaggedFraud"],
        margins=True,
    )

    print("\n3. ISFRAUD VS ISFLAGGEDFRAUD")
    print(flag_comparison)

    flagged_actual_fraud = df.loc[
        df["isFlaggedFraud"] == 1,
        "isFraud",
    ].sum()

    total_flagged = df["isFlaggedFraud"].sum()

    print(f"\nTransactions flagged by existing rule: {total_flagged:,}")
    print(f"Flagged transactions that are fraud:  {flagged_actual_fraud:,}")

    # Time-step information
    print("\n4. TIME-STEP INFORMATION")
    print(f"Minimum step: {df['step'].min()}")
    print(f"Maximum step: {df['step'].max()}")
    print(f"Unique steps: {df['step'].nunique()}")

    # Fraud amount analysis
    fraud_amounts = df.loc[df["isFraud"] == 1, "amount"]
    legitimate_amounts = df.loc[df["isFraud"] == 0, "amount"]

    amount_summary = pd.DataFrame(
        {
            "fraud": fraud_amounts.describe(),
            "non_fraud": legitimate_amounts.describe(),
        }
    )

    print("\n5. TRANSACTION AMOUNT SUMMARY")
    print(amount_summary)

    # Fraud count per time step
    fraud_by_step = (
        df.loc[df["isFraud"] == 1]
        .groupby("step")
        .size()
        .sort_values(ascending=False)
    )

    print("\n6. TOP 10 STEPS WITH MOST FRAUD")
    print(fraud_by_step.head(10))

    print("\n" + "=" * 70)
    print("ANALYSIS COMPLETED")
    print("=" * 70)

In [ ]:
analyse_fraud(df)


PAYSIM FRAUD ANALYSIS

1. FRAUD COUNTS BY TRANSACTION TYPE
isFraud   non_fraud  fraud      All
type                               
CASH_IN     1399284      0  1399284
CASH_OUT    2233384   4116  2237500
DEBIT         41432      0    41432
PAYMENT     2151495      0  2151495
TRANSFER     528812   4097   532909
All         6354407   8213  6362620

2. FRAUD RATE BY TRANSACTION TYPE
          total_transactions  fraud_transactions  fraud_percentage
type                                                              
CASH_OUT             2237500                4116          0.183955
TRANSFER              532909                4097          0.768799
CASH_IN              1399284                   0          0.000000
DEBIT                  41432                   0          0.000000
PAYMENT              2151495                   0          0.000000

3. ISFRAUD VS ISFLAGGEDFRAUD
isFlaggedFraud        0   1      All
isFraud                             
0               6354407   0  6354407
1      

In [ ]:
def engineer_features(df):
    df = df.copy()

    # Time features
    df["hour"] = (df["step"] - 1) % 24
    df["day"] = (df["step"] - 1) // 24

    # Amount and balance features
    df["log_amount"] = np.log1p(df["amount"])

    df["amount_to_origin_balance"] = (
        df["amount"] / (df["oldbalanceOrg"] + 1)
    )

    df["amount_to_destination_balance"] = (
        df["amount"] / (df["oldbalanceDest"] + 1)
    )

    df["amount_exceeds_origin_balance"] = (
        df["amount"] > df["oldbalanceOrg"]
    ).astype(int)

    df["origin_balance_zero"] = (
        df["oldbalanceOrg"] == 0
    ).astype(int)

    df["destination_balance_zero"] = (
        df["oldbalanceDest"] == 0
    ).astype(int)

    # Encode transaction type
    df = pd.get_dummies(
        df,
        columns=["type"],
        dtype=int
    )

    # Target
    y = df["isFraud"].copy()

    # Remove identifiers, target, leakage, and post-transaction columns
    X = df.drop(
        columns=[
            "nameOrig",
            "nameDest",
            "isFraud",
            "isFlaggedFraud",
            "newbalanceOrig",
            "newbalanceDest",
        ]
    )

    return X, y

X, y = engineer_features(df)

In [ ]:
def train_val_test_split(
    X,
    y,
    train_size=0.70,
    validation_size=0.15,
):
    test_size = 1 - train_size - validation_size

    if train_size <= 0 or validation_size <= 0 or test_size <= 0:
        raise ValueError("Train, validation, and test sizes must be positive.")

    # Ensure transactions are ordered chronologically
    sorted_indices = X["step"].sort_values().index
    X = X.loc[sorted_indices]
    y = y.loc[sorted_indices]

    # Split training data from validation + test data
    X_train, X_remaining, y_train, y_remaining = train_test_split(
        X,
        y,
        train_size=train_size,
        shuffle=False,
    )

    # Calculate validation proportion within the remaining data
    relative_validation_size = validation_size / (
        validation_size + test_size
    )

    X_validation, X_test, y_validation, y_test = train_test_split(
        X_remaining,
        y_remaining,
        train_size=relative_validation_size,
        shuffle=False,
    )

    return (
        X_train,
        X_validation,
        X_test,
        y_train,
        y_validation,
        y_test,)

X_train, X_validation, X_test, y_train, y_validation, y_test = train_val_test_split(X, y)
    

In [ ]:
def check_split_distribution(y_train, y_validation, y_test):
    splits = {
        "Train": y_train,
        "Validation": y_validation,
        "Test": y_test,
    }

    for name, y_split in splits.items():
        total = len(y_split)
        fraud = int(y_split.sum())
        fraud_rate = (fraud / total) * 100

        print(
            f"{name}: "
            f"rows={total:,}, "
            f"fraud={fraud:,}, "
            f"fraud_rate={fraud_rate:.6f}%"
        )

check_split_distribution(
    y_train,
    y_validation,
    y_test,
)

Train: rows=4,453,834, fraud=3,643, fraud_rate=0.081795%
Validation: rows=954,392, fraud=562, fraud_rate=0.058886%
Test: rows=954,394, fraud=4,008, fraud_rate=0.419952%
